# Nvidia Nemotron Reasoning Challenge
## Score: .53

In [ ]:
import subprocess, sys, os
from pathlib import Path
def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read()
            rel_pack_path = (pth_file.parent/relpath)
            if rel_pack_path.exists():
                print(f"append {rel_pack_path}")
                sys.path.append(str(rel_pack_path))


offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir = "/kaggle/working/packages"

os.makedirs(target_dir, exist_ok=True)
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl"
    ])
    print("Installed from offline packages")

sys.path.append(target_dir)
resolve_python_path(target_dir)

import datasets, cutlass

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys, stat, shutil, gc, zipfile, re
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset, concatenate_datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. In Kaggle, set Accelerator to GPU before running training.")
print(f"torch={torch.__version__} | cuda={torch.cuda.is_available()} | gpu={torch.cuda.get_device_name(0)}")


In [ ]:
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None: out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")

In [ ]:
SAMPLE_SIZE = 450
SAMPLE_SEED = 42
LORA_RANK = 32
LORA_ALPHA = 32
# Menu 6: LoRA target strategy — "attn_only" adapts attention projections only (less MLP memorization).
LORA_TARGET_MODE = "attn_only"
MAX_SEQ_LEN = 2048
NUM_EPOCHS = 1.25
STAGE_B_EPOCHS = 0.15
STAGE_B_MAX_ROWS = 500
# Menu 3: competition-aligned synthetic rows from official train (instruction variants + short template trace).
MENU3_OFFICIAL_SYNTH = True
MENU3_MAX_BASE_ROWS = 350
MENU3_VARIANTS_PER_PROMPT = 2
MENU3_INSTR_HINTS = [
    "\nThink step by step, then put your final answer inside \\boxed{}.",
    "\nCheck edge cases carefully. Final answer in \\boxed{}.",
    "\nGive a short justification, then \\boxed{}.",
]
# Menu 4: verify-then-box pattern on official train (template self-correction, no extra LLM calls).
MENU4_SELF_CORRECT = True
MENU4_MAX_BASE_ROWS = 140
MENU4_VARIANTS_PER_PROMPT = 2
MENU4_USER_HINTS = [
    "\nAfter solving, double-check your work, then put the final answer in \\boxed{}.",
    "\nInclude a brief self-check before giving \\boxed{}.",
]
# Menu 5: hard-negative contrast (integer answers only — synthetic wrong candidate).
MENU5_HARD_NEGATIVE = True
MENU5_MAX_ROWS = 180
BATCH_SIZE = 1
GRAD_ACCUM = 6
LR = 1e-4
STAGE_B_LR = 4e-5
LORA_DROPOUT = 0.05
# Stage A supervision mode (plateau escape): "boxed_only" = train metric-shaped targets only;
# "cot_tail" = keep truncated CoT + boxed (previous experiment).
STAGE_A_SUPERVISION = "boxed_only"
STAGE_A_COT_TAIL_CHARS = 1400
STAGE_A_DEDUPE_PROMPT_ANSWER = True
STAGE_B_BOXED_ONLY_ASSISTANT = True
WEIGHT_DECAY = 0.01
RUN_NAME = (
    f"run_menu3456_{STAGE_A_SUPERVISION}_"
    f"lora{LORA_TARGET_MODE}_"
    f"a{NUM_EPOCHS}_b{STAGE_B_EPOCHS}_s{SAMPLE_SIZE}_bcap{STAGE_B_MAX_ROWS}"
    f"_m3b{MENU3_MAX_BASE_ROWS}v{MENU3_VARIANTS_PER_PROMPT}"
    f"_m4b{MENU4_MAX_BASE_ROWS}v{MENU4_VARIANTS_PER_PROMPT}"
    f"_m5n{MENU5_MAX_ROWS}"
)
OUTPUT_DIR = f"/kaggle/working/adapter_{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

_MODEL_CANDIDATES = [
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default",
]
MODEL_PATH = None
for _mp in _MODEL_CANDIDATES:
    if os.path.isdir(_mp):
        MODEL_PATH = _mp
        print(f"Using local model path: {MODEL_PATH}")
        break
if MODEL_PATH is None:
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Using kagglehub model path: {MODEL_PATH}")

STAGE_A_DATA_PATH = "/kaggle/input/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv"
OFFICIAL_TRAIN_PATH = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"

if not os.path.isfile(STAGE_A_DATA_PATH):
    raise FileNotFoundError(f"Missing Stage A dataset file: {STAGE_A_DATA_PATH}")
if not os.path.isfile(OFFICIAL_TRAIN_PATH):
    raise FileNotFoundError(f"Missing official train.csv: {OFFICIAL_TRAIN_PATH}")

print(f"Using Stage A data: {STAGE_A_DATA_PATH}")
print(f"Using official train.csv: {OFFICIAL_TRAIN_PATH}")
print(f"Stage A supervision mode: {STAGE_A_SUPERVISION}")
stage_a_df = pl.read_csv(STAGE_A_DATA_PATH)
stage_a_df = stage_a_df.with_columns(
    pl.col("prompt").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
    pl.col("answer").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
    pl.col("generated_cot").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
)
if STAGE_A_SUPERVISION == "boxed_only":
    stage_a_df = stage_a_df.filter(
        pl.col("prompt").str.len_chars() > 0,
        pl.col("answer").str.len_chars() > 0,
    )
else:
    stage_a_df = stage_a_df.filter(
        pl.col("prompt").str.len_chars() > 0,
        pl.col("answer").str.len_chars() > 0,
        pl.col("generated_cot").str.len_chars() > 0,
    )

if STAGE_A_DEDUPE_PROMPT_ANSWER:
    _n = len(stage_a_df)
    stage_a_df = stage_a_df.unique(subset=["prompt", "answer"], keep="first")
    print(f"Stage A dedupe (prompt,answer): {_n} -> {len(stage_a_df)}")

n_pool = len(stage_a_df)
take = min(SAMPLE_SIZE, n_pool)
if take < n_pool:
    stage_a_df = stage_a_df.sample(n=take, seed=SAMPLE_SEED)

stage_b_df = pl.read_csv(OFFICIAL_TRAIN_PATH)
stage_b_df = stage_b_df.with_columns(
    pl.col("prompt").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
    pl.col("answer").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
)
stage_b_df = stage_b_df.filter(
    pl.col("prompt").str.len_chars() > 0,
    pl.col("answer").str.len_chars() > 0,
)

stage_b_pool = len(stage_b_df)
if STAGE_B_MAX_ROWS is not None:
    stage_b_take = min(STAGE_B_MAX_ROWS, stage_b_pool)
    if stage_b_take < stage_b_pool:
        stage_b_df = stage_b_df.sample(n=stage_b_take, seed=SAMPLE_SEED)

menu3_df = None
if MENU3_OFFICIAL_SYNTH and len(stage_b_df) > 0:
    m3_take = min(MENU3_MAX_BASE_ROWS, len(stage_b_df))
    if m3_take < len(stage_b_df):
        menu3_base = stage_b_df.sample(n=m3_take, seed=SAMPLE_SEED + 9)
    else:
        menu3_base = stage_b_df
    _hints = MENU3_INSTR_HINTS[: max(1, MENU3_VARIANTS_PER_PROMPT)]
    menu3_df = menu3_base.join(pl.DataFrame({"instr_hint": _hints}), how="cross")

menu4_df = None
if MENU4_SELF_CORRECT and len(stage_b_df) > 0:
    m4_take = min(MENU4_MAX_BASE_ROWS, len(stage_b_df))
    menu4_base = stage_b_df.sample(n=m4_take, seed=SAMPLE_SEED + 11)
    _m4h = MENU4_USER_HINTS[: max(1, MENU4_VARIANTS_PER_PROMPT)]
    menu4_df = menu4_base.join(pl.DataFrame({"m4_hint": _m4h}), how="cross")

def _menu5_wrong_int_candidate(ans: str):
    s = (ans or "").strip()
    if not re.fullmatch(r"-?\d+", s):
        return None
    v = int(s)
    for delta in (1, -1, 2, -2, 7, -7, 13, -13):
        w = v + delta
        if str(w) != s:
            return str(w)
    return str(v + 999)

menu5_df = None
if MENU5_HARD_NEGATIVE and len(stage_b_df) > 0:
    overshoot = min(max(MENU5_MAX_ROWS * 4, MENU5_MAX_ROWS), len(stage_b_df))
    m5_scan = stage_b_df.sample(n=overshoot, seed=SAMPLE_SEED + 13)
    m5_rows = []
    for row in m5_scan.iter_rows(named=True):
        wrong = _menu5_wrong_int_candidate(row.get("answer", ""))
        if wrong is None:
            continue
        m5_rows.append(
            {
                "prompt": row["prompt"],
                "answer": row["answer"],
                "wrong_candidate": wrong,
            }
        )
        if len(m5_rows) >= MENU5_MAX_ROWS:
            break
    if m5_rows:
        menu5_df = pl.DataFrame(m5_rows)

print(f"Stage A subset: {len(stage_a_df)} rows (pool {n_pool}, cap {SAMPLE_SIZE})")
stage_b_rows = len(stage_b_df)
print(f"Stage B official rows: {stage_b_rows} (pool {stage_b_pool}, cap {STAGE_B_MAX_ROWS})")
if menu3_df is not None:
    print(f"Menu3 synthetic rows (official x variants): {len(menu3_df)}")
if menu4_df is not None:
    print(f"Menu4 self-correct rows (official x variants): {len(menu4_df)}")
if menu5_df is not None:
    print(f"Menu5 hard-negative rows: {len(menu5_df)}")
print(f"Run: {RUN_NAME} | stage_a_rows={len(stage_a_df)} | stage_b_rows={stage_b_rows} | MAX_SEQ_LEN={MAX_SEQ_LEN} | output: {OUTPUT_DIR}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def apply_template(user_msg, assistant_msg):
    try:
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        return (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        )

def build_stage_a_text(example):
    prompt = example["prompt"]
    answer = example["answer"]
    user_msg = prompt + "\nPut your final answer inside \\boxed{}."
    if STAGE_A_SUPERVISION == "boxed_only":
        assistant_msg = f"\\boxed{{{answer}}}"
    else:
        cot = example["generated_cot"]
        if STAGE_A_COT_TAIL_CHARS and len(cot) > STAGE_A_COT_TAIL_CHARS:
            cot = cot[-STAGE_A_COT_TAIL_CHARS:]
        assistant_msg = f"{cot}\n\n\\boxed{{{answer}}}"
    return {"text": apply_template(user_msg, assistant_msg)}

def build_stage_b_text(example):
    prompt = example["prompt"]
    answer = example["answer"]

    user_msg = prompt + "\nSolve carefully and put your final answer inside \\boxed{}."
    if STAGE_B_BOXED_ONLY_ASSISTANT:
        assistant_msg = f"\\boxed{{{answer}}}"
    else:
        assistant_msg = f"The final answer is \\boxed{{{answer}}}."
    return {"text": apply_template(user_msg, assistant_msg)}

def build_menu3_text(example):
    """Menu item 3: official prompts + varied instructions + short template trace ending in boxed."""
    prompt = example["prompt"]
    answer = example["answer"]
    hint = example["instr_hint"]
    user_msg = prompt + hint
    assistant_msg = (
        "Working through the problem systematically:\n"
        "- Extract what needs to be computed from the prompt.\n"
        "- Apply the stated rules without skipping steps.\n"
        "- Sanity-check consistency with the setup.\n\n"
        f"\\boxed{{{answer}}}"
    )
    return {"text": apply_template(user_msg, assistant_msg)}

def build_menu4_text(example):
    """Menu item 4: draft + explicit self-check + final boxed (template distillation)."""
    prompt = example["prompt"]
    answer = example["answer"]
    hint = example["m4_hint"]
    user_msg = prompt + hint
    assistant_msg = (
        "Draft:\n"
        "- Parse the problem and carry the main computation.\n\n"
        "Self-check:\n"
        "- Re-read the prompt for missed constraints or misread quantities.\n"
        "- Recompute the critical step if anything looks inconsistent.\n"
        "- Confirm the result matches the problem's required form.\n\n"
        f"Final: \\boxed{{{answer}}}"
    )
    return {"text": apply_template(user_msg, assistant_msg)}

def build_menu5_text(example):
    """Menu item 5: reject a wrong integer candidate, then emit correct boxed answer."""
    prompt = example["prompt"]
    answer = example["answer"]
    wrong = example["wrong_candidate"]
    user_msg = (
        prompt
        + f"\nA classmate claims the answer is: {wrong}\n"
        + "If that is incorrect, say so briefly and give the correct final answer in \\boxed{}."
    )
    assistant_msg = (
        f"The claim {wrong} is incorrect for this problem.\n"
        f"The correct answer is \\boxed{{{answer}}}"
    )
    return {"text": apply_template(user_msg, assistant_msg)}

hf_dataset_stage_a = Dataset.from_pandas(stage_a_df.to_pandas()).map(
    build_stage_a_text,
    remove_columns=stage_a_df.columns,
)
if stage_b_df is not None and len(stage_b_df) > 0:
    hf_dataset_stage_b = Dataset.from_pandas(stage_b_df.to_pandas()).map(
        build_stage_b_text,
        remove_columns=stage_b_df.columns,
    )
    if menu3_df is not None and len(menu3_df) > 0:
        hf_menu3 = Dataset.from_pandas(menu3_df.to_pandas()).map(
            build_menu3_text,
            remove_columns=menu3_df.columns,
        )
        hf_dataset_stage_b = concatenate_datasets([hf_dataset_stage_b, hf_menu3])
        print(f"Menu3 rows merged into Stage B: {len(hf_menu3)}")
    if menu4_df is not None and len(menu4_df) > 0:
        hf_menu4 = Dataset.from_pandas(menu4_df.to_pandas()).map(
            build_menu4_text,
            remove_columns=menu4_df.columns,
        )
        hf_dataset_stage_b = concatenate_datasets([hf_dataset_stage_b, hf_menu4])
        print(f"Menu4 rows merged into Stage B: {len(hf_menu4)}")
    if menu5_df is not None and len(menu5_df) > 0:
        hf_menu5 = Dataset.from_pandas(menu5_df.to_pandas()).map(
            build_menu5_text,
            remove_columns=menu5_df.columns,
        )
        hf_dataset_stage_b = concatenate_datasets([hf_dataset_stage_b, hf_menu5])
        print(f"Menu5 rows merged into Stage B: {len(hf_menu5)}")
else:
    hf_dataset_stage_b = None

print(f"Tokenized Stage A rows: {len(hf_dataset_stage_a)}")
print(f"Tokenized Stage B rows: {len(hf_dataset_stage_b) if hf_dataset_stage_b is not None else 0}")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
model.gradient_checkpointing_enable()

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

def _resolve_lora_target_modules(base_model, mode: str):
    if mode == "all_linear":
        return "all-linear"
    if mode != "attn_only":
        raise ValueError(f"Unknown LORA_TARGET_MODE: {mode}")
    watch = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "qkv_proj", "wq", "wk", "wv", "wo",
        "linear_q", "linear_k", "linear_v", "linear_proj",
    )
    tails = set()
    for mod_name, _mod in base_model.named_modules():
        tail = mod_name.rsplit(".", 1)[-1]
        if tail in watch:
            tails.add(tail)
    for group in (
        ("q_proj", "k_proj", "v_proj", "o_proj"),
        ("wq", "wk", "wv", "wo"),
        ("linear_q", "linear_k", "linear_v", "linear_proj"),
    ):
        if all(t in tails for t in group):
            return list(group)
    if tails:
        out = sorted(tails)
        print(f"LoRA attn_only: using discovered tails {out}")
        return out
    print("WARNING: attn_only found no layers; falling back to all-linear")
    return "all-linear"

LORA_TARGET_MODULES = _resolve_lora_target_modules(model, LORA_TARGET_MODE)
print(f"LoRA target_modules ({LORA_TARGET_MODE}): {LORA_TARGET_MODULES}")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

def build_training_args(stage_name, num_epochs, learning_rate):
    return SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        logging_steps=5,
        bf16=True,
        max_grad_norm=1.0,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        weight_decay=WEIGHT_DECAY,
        save_strategy="no",
        report_to="none",
        run_name=f"{RUN_NAME}_{stage_name}",
        dataset_text_field="text",
        max_length=MAX_SEQ_LEN,
        packing=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

print("Starting Stage A on external CoT data...")
stage_a_trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset_stage_a,
    processing_class=tokenizer,
    args=build_training_args("stage_a", NUM_EPOCHS, LR),
)
stage_a_trainer.train()
model = stage_a_trainer.model

gc.collect()
torch.cuda.empty_cache()

if hf_dataset_stage_b is not None and len(hf_dataset_stage_b) > 0 and STAGE_B_EPOCHS > 0:
    print("Starting Stage B on official competition train.csv...")
    stage_b_trainer = SFTTrainer(
        model=model,
        train_dataset=hf_dataset_stage_b,
        processing_class=tokenizer,
        args=build_training_args("stage_b", STAGE_B_EPOCHS, STAGE_B_LR),
    )
    stage_b_trainer.train()
    stage_b_trainer.model.save_pretrained(OUTPUT_DIR)
else:
    print("Skipping Stage B (disabled or empty dataset).")
    model.save_pretrained(OUTPUT_DIR)

print(f"Adapter saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

In [ ]:
# Pre-submit sanity checks (run before clicking Submit)
import os

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: GPU is disabled. Training cells will fail before creating submission.zip")

required_adapter_files = ["adapter_config.json", "adapter_model.safetensors"]
missing = [f for f in required_adapter_files if not os.path.isfile(os.path.join(OUTPUT_DIR, f))]
if missing:
    raise FileNotFoundError(
        f"Missing adapter files in {OUTPUT_DIR}: {missing}. Run training cell first."
    )

print("Adapter files exist:")
for f in required_adapter_files:
    p = os.path.join(OUTPUT_DIR, f)
    print(f"  {p} ({os.path.getsize(p)/1024/1024:.2f} MB)")

zip_path = "/kaggle/working/submission.zip"
if os.path.isfile(zip_path):
    print(f"Found existing {zip_path} ({os.path.getsize(zip_path)/1024/1024:.2f} MB)")
else:
    print(f"No existing {zip_path} yet. Run packaging cell next.")

In [ ]:
import zipfile
import os

zip_path = "/kaggle/working/submission.zip"

print(f"Packaging files from {OUTPUT_DIR}...")


REQUIRED_ZIP = {"adapter_config.json", "adapter_model.safetensors"}
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in REQUIRED_ZIP:
        fpath = os.path.join(OUTPUT_DIR, fname)
        if not os.path.isfile(fpath):
            raise FileNotFoundError(f"Missing {fpath}")
        zf.write(fpath, arcname=fname)

print(f"Created {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")

with zipfile.ZipFile(zip_path, 'r') as zf:
    zip_contents = zf.namelist()
    print(f"Zip Contents: {zip_contents}")
    
    if "adapter_config.json" not in zip_contents:
        raise AssertionError("CRITICAL ERROR: adapter_config.json is missing from the zip. The Kaggle evaluation will fail.")
    if "adapter_model.safetensors" not in zip_contents:
         raise AssertionError("CRITICAL ERROR: adapter_model.safetensors is missing from the zip.")

print("submission.zip is ready")